In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import optuna
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import ElasticNet
from sklearn.decomposition import PCA
import os
from datetime import datetime

/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset
file_path = "kacper_reg_739582(1).csv"  # Update this with your file path
data = pd.read_csv(file_path)

In [3]:
data.shape

(73349, 22)

In [4]:
data.columns

Index(['bin_resp_1', 'bin_resp_2', 'bin_resp_3', 'bin_resp_4', 'bin_resp_5',
       'bin_base_1', 'bin_base_2', 'bin_base_3', 'bin_base_4', 'bin_base_5',
       'mean_sniff_resp', 'mean_sniff_baseline', 'chem_id', 'uch', 'occur',
       'novelty', 'familiarity', 'ic', 'cell_count', 'region', 'animal',
       'uid'],
      dtype='object')

In [5]:
label_encoder = LabelEncoder()
data['uch_encoded'] = label_encoder.fit_transform(data['uch'])

In [6]:
data = data.drop(columns=['cell_count'])
data = data.drop(columns=['uid'], errors='ignore')
data = data.drop(columns=['chem_id'], errors='ignore')
data = data.drop(columns=['animal'], errors='ignore')
data = data.drop(columns=['novelty'], errors='ignore')

In [7]:
data.columns

Index(['bin_resp_1', 'bin_resp_2', 'bin_resp_3', 'bin_resp_4', 'bin_resp_5',
       'bin_base_1', 'bin_base_2', 'bin_base_3', 'bin_base_4', 'bin_base_5',
       'mean_sniff_resp', 'mean_sniff_baseline', 'uch', 'occur', 'familiarity',
       'ic', 'region', 'uch_encoded'],
      dtype='object')

In [8]:
# Split data into features (X) and target (y)
X = data.drop(columns=['uch', 'uch_encoded'], errors='ignore')
y = data['uch_encoded']

In [9]:
X.columns

Index(['bin_resp_1', 'bin_resp_2', 'bin_resp_3', 'bin_resp_4', 'bin_resp_5',
       'bin_base_1', 'bin_base_2', 'bin_base_3', 'bin_base_4', 'bin_base_5',
       'mean_sniff_resp', 'mean_sniff_baseline', 'occur', 'familiarity', 'ic',
       'region'],
      dtype='object')

In [10]:
y

0        15
1        15
2        15
3        15
4        15
         ..
73344    19
73345     0
73346     0
73347     0
73348     0
Name: uch_encoded, Length: 73349, dtype: int64

In [11]:
# Convert categorical columns in X to numerical
X = pd.get_dummies(X, drop_first=True)

In [12]:
# Step 1: Preprocessing
# Standardize numerical features
numerical_columns = data.select_dtypes(include=['int64', 'float64']).columns
scaler = StandardScaler()
data[numerical_columns] = scaler.fit_transform(data[numerical_columns])

In [13]:
# Check for non-numeric columns in X
non_numeric_cols = X.select_dtypes(exclude=[np.number]).columns

if not non_numeric_cols.empty:
    print(f"Non-numeric columns found: {non_numeric_cols}")
    for col in non_numeric_cols:
        print(f"Unique values in '{col}': {X[col].unique()}")

    raise ValueError("Some features are still non-numeric after encoding.")

# Verify all columns are numeric
if not all(X.dtypes.apply(lambda x: np.issubdtype(x, np.number))):
    raise ValueError("Some features are still non-numeric after encoding.")

In [14]:
# Step 2: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [15]:
# Optuna optimization for Random Forest

def objective_rf(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [16]:
# Optuna optimization for Logistic Regression

def objective_lr(trial):
    C = trial.suggest_float("C", 0.01, 10.0, log=True)
    
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [17]:
# Optuna optimization for Support Vector Machine

def objective_svc(trial):
    C = trial.suggest_float("C", 0.01, 10.0, log=True)
    kernel = trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"])
    
    model = SVC(C=C, kernel=kernel, probability=True, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [18]:
def objective_knn(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    
    model = KNeighborsClassifier(n_neighbors=n_neighbors)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [19]:
# Optuna optimization for Naive Bayes
def objective_nb(trial):
    model = GaussianNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [20]:
# Optuna optimization for Decision Tree
def objective_dt(trial):
    max_depth = trial.suggest_int("max_depth", 2, 20)
    
    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [21]:
# Optuna optimization for Gradient Boosting
def objective_gb(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    
    model = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [22]:
# Optuna optimization for LightGBM
def objective_lgb(trial):
    num_classes = len(np.unique(y_train))  # Calculate the number of unique classes
    params = {
        "objective": "multiclass",
        "metric": "multi_logloss",
        "verbosity": -1,
        "boosting_type": trial.suggest_categorical("boosting_type", ["gbdt", "dart"]),
        "num_leaves": trial.suggest_int("num_leaves", 16, 64),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "n_estimators": trial.suggest_int("n_estimators", 10, 200),
        "num_class": num_classes  # Specify the number of classes
    }
    
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [23]:
# Optuna optimization for XGBoost
def objective_xgb(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "use_label_encoder": False,
        "n_estimators": trial.suggest_int("n_estimators", 10, 200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "max_depth": trial.suggest_int("max_depth", 2, 20),
    }
    
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [24]:
# Optuna optimization for CatBoost
def objective_cb(trial):
    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "iterations": trial.suggest_int("iterations", 10, 200),
    }
    
    model = CatBoostClassifier(**params, verbose=0)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

In [25]:
# Optuna optimization for ElasticNet
def objective_en(trial):
    alpha = trial.suggest_float("alpha", 0.01, 10.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)
    
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
    model.fit(X_train, y_train)
    y_pred = np.round(model.predict(X_test))  # Round predictions for classification
    return accuracy_score(y_test, y_pred)

In [26]:
# Run Optuna for each model
print("Optimizing Random Forest...")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=50)
print("Best parameters for Random Forest:", study_rf.best_params)
print("Best accuracy for Random Forest:", study_rf.best_value)

[I 2025-05-20 00:22:05,651] A new study created in memory with name: no-name-44154cf4-6144-425c-81b0-3bad3a17df43


Optimizing Random Forest...


[I 2025-05-20 00:22:16,295] Trial 0 finished with value: 0.526198591229266 and parameters: {'n_estimators': 142, 'max_depth': 13}. Best is trial 0 with value: 0.526198591229266.
[I 2025-05-20 00:22:23,324] Trial 1 finished with value: 0.38264030902067714 and parameters: {'n_estimators': 124, 'max_depth': 9}. Best is trial 0 with value: 0.526198591229266.
[I 2025-05-20 00:22:32,934] Trial 2 finished with value: 0.41981367870938424 and parameters: {'n_estimators': 167, 'max_depth': 10}. Best is trial 0 with value: 0.526198591229266.
[I 2025-05-20 00:22:37,413] Trial 3 finished with value: 0.26180413542376735 and parameters: {'n_estimators': 177, 'max_depth': 4}. Best is trial 0 with value: 0.526198591229266.
[I 2025-05-20 00:22:49,520] Trial 4 finished with value: 0.5566916609861395 and parameters: {'n_estimators': 177, 'max_depth': 14}. Best is trial 4 with value: 0.5566916609861395.
[I 2025-05-20 00:22:59,265] Trial 5 finished with value: 0.41863212906157693 and parameters: {'n_estimat

Best parameters for Random Forest: {'n_estimators': 123, 'max_depth': 20}
Best accuracy for Random Forest: 0.6565326062258577


In [27]:
print("Optimizing Logistic Regression...")
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_lr, n_trials=50)
print("Best parameters for Logistic Regression:", study_lr.best_params)
print("Best accuracy for Logistic Regression:", study_lr.best_value)

[I 2025-05-20 00:28:35,847] A new study created in memory with name: no-name-8dae9e5c-701f-4874-9613-0e11f7d8defd


Optimizing Logistic Regression...


/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[I 2025-05-20 00:30:24,973] Trial 0 finished with value: 0.21890479436491705 and parameters: {'C': 0.03225180491278425}. Best is trial 0 with value: 0.21890479436491705.
/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://sciki

Best parameters for Logistic Regression: {'C': 0.04012806090731415}
Best accuracy for Logistic Regression: 0.2190865712338105


In [28]:
print("Optimizing k-Nearest Neighbors...")
study_knn = optuna.create_study(direction="maximize")
study_knn.optimize(objective_knn, n_trials=50)
print("Best parameters for k-Nearest Neighbors:", study_knn.best_params)
print("Best accuracy for k-Nearest Neighbors:", study_knn.best_value)

[I 2025-05-20 01:49:45,466] A new study created in memory with name: no-name-dd765016-231c-4a1f-8f1f-9a8678125ff5


Optimizing k-Nearest Neighbors...


[I 2025-05-20 01:49:50,602] Trial 0 finished with value: 0.1314246762099523 and parameters: {'n_neighbors': 44}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:49:54,891] Trial 1 finished with value: 0.09725062485798681 and parameters: {'n_neighbors': 6}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:49:59,416] Trial 2 finished with value: 0.12379004771642808 and parameters: {'n_neighbors': 32}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:50:04,213] Trial 3 finished with value: 0.11838218586684844 and parameters: {'n_neighbors': 24}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:50:08,797] Trial 4 finished with value: 0.1113383321972279 and parameters: {'n_neighbors': 16}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:50:13,744] Trial 5 finished with value: 0.12992501704158146 and parameters: {'n_neighbors': 41}. Best is trial 0 with value: 0.1314246762099523.
[I 2025-05-20 01:50:18,118] Tri

Best parameters for k-Nearest Neighbors: {'n_neighbors': 50}
Best accuracy for k-Nearest Neighbors: 0.13483299250170416


In [29]:
print("Optimizing Naive Bayes...")
study_nb = optuna.create_study(direction="maximize")
study_nb.optimize(objective_nb, n_trials=50)
print("Best accuracy for Naive Bayes:", study_nb.best_value)

[I 2025-05-20 01:53:46,949] A new study created in memory with name: no-name-b853fb5a-0d9c-427c-af99-521c0b6092b0
[I 2025-05-20 01:53:47,029] Trial 0 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,097] Trial 1 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.


Optimizing Naive Bayes...


[I 2025-05-20 01:53:47,174] Trial 2 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,242] Trial 3 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,301] Trial 4 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,367] Trial 5 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,435] Trial 6 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,514] Trial 7 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value: 0.19341058850261303.
[I 2025-05-20 01:53:47,585] Trial 8 finished with value: 0.19341058850261303 and parameters: {}. Best is trial 0 with value:

Best accuracy for Naive Bayes: 0.19341058850261303


In [30]:
print("Optimizing Decision Tree...")
study_dt = optuna.create_study(direction="maximize")
study_dt.optimize(objective_dt, n_trials=50)
print("Best parameters for Decision Tree:", study_dt.best_params)
print("Best accuracy for Decision Tree:", study_dt.best_value)

[I 2025-05-20 01:53:50,523] A new study created in memory with name: no-name-6e372914-6f28-452f-a6a8-e01f786be570
[I 2025-05-20 01:53:50,645] Trial 0 finished with value: 0.23767325607816406 and parameters: {'max_depth': 5}. Best is trial 0 with value: 0.23767325607816406.


Optimizing Decision Tree...


[I 2025-05-20 01:53:50,721] Trial 1 finished with value: 0.2106793910474892 and parameters: {'max_depth': 3}. Best is trial 0 with value: 0.23767325607816406.
[I 2025-05-20 01:53:50,815] Trial 2 finished with value: 0.2162235855487389 and parameters: {'max_depth': 4}. Best is trial 0 with value: 0.23767325607816406.
[I 2025-05-20 01:53:50,941] Trial 3 finished with value: 0.25548738922972053 and parameters: {'max_depth': 6}. Best is trial 3 with value: 0.25548738922972053.
[I 2025-05-20 01:53:51,290] Trial 4 finished with value: 0.6593501476937059 and parameters: {'max_depth': 20}. Best is trial 4 with value: 0.6593501476937059.
[I 2025-05-20 01:53:51,564] Trial 5 finished with value: 0.5499659168370825 and parameters: {'max_depth': 15}. Best is trial 4 with value: 0.6593501476937059.
[I 2025-05-20 01:53:51,905] Trial 6 finished with value: 0.6593501476937059 and parameters: {'max_depth': 20}. Best is trial 4 with value: 0.6593501476937059.
[I 2025-05-20 01:53:52,053] Trial 7 finished 

Best parameters for Decision Tree: {'max_depth': 20}
Best accuracy for Decision Tree: 0.6593501476937059


In [39]:
print("Optimizing Gradient Boosting...")
study_gb = optuna.create_study(direction="maximize")
study_gb.optimize(objective_gb, n_trials=50)
print("Best parameters for Gradient Boosting:", study_gb.best_params)
print("Best accuracy for Gradient Boosting:", study_gb.best_value)

[I 2025-05-20 05:33:43,819] A new study created in memory with name: no-name-5d6ad650-85d0-41c2-a401-e8113e4311e3


Optimizing Gradient Boosting...


[I 2025-05-20 05:37:28,921] Trial 0 finished with value: 0.5773233356055442 and parameters: {'n_estimators': 191, 'learning_rate': 0.162992045726471}. Best is trial 0 with value: 0.5773233356055442.
[I 2025-05-20 05:40:55,519] Trial 1 finished with value: 0.6075437400590775 and parameters: {'n_estimators': 178, 'learning_rate': 0.2494240978457308}. Best is trial 1 with value: 0.6075437400590775.
[I 2025-05-20 05:43:06,942] Trial 2 finished with value: 0.4991592819813679 and parameters: {'n_estimators': 112, 'learning_rate': 0.1158065122851073}. Best is trial 1 with value: 0.6075437400590775.
[I 2025-05-20 05:43:38,590] Trial 3 finished with value: 0.29811406498523063 and parameters: {'n_estimators': 28, 'learning_rate': 0.054689970531376954}. Best is trial 1 with value: 0.6075437400590775.
[I 2025-05-20 05:44:57,383] Trial 4 finished with value: 0.4185412406271302 and parameters: {'n_estimators': 69, 'learning_rate': 0.08436421457531465}. Best is trial 1 with value: 0.6075437400590775.

KeyboardInterrupt: 

In [32]:
print("Optimizing LightGBM...")
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=50)
print("Best parameters for LightGBM:", study_lgb.best_params)
print("Best accuracy for LightGBM:", study_lgb.best_value)

[I 2025-05-20 04:26:50,283] A new study created in memory with name: no-name-5f32232e-471b-476b-9656-12d6ac704ce1


Optimizing LightGBM...


[I 2025-05-20 04:26:51,508] Trial 0 finished with value: 0.535196546239491 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 17, 'learning_rate': 0.08540438856443358, 'n_estimators': 22}. Best is trial 0 with value: 0.535196546239491.
[I 2025-05-20 04:27:01,654] Trial 1 finished with value: 0.6976141785957737 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 23, 'learning_rate': 0.17408284857928877, 'n_estimators': 190}. Best is trial 1 with value: 0.6976141785957737.
[I 2025-05-20 04:27:08,111] Trial 2 finished with value: 0.6937514201317883 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 40, 'learning_rate': 0.12161958257656673, 'n_estimators': 91}. Best is trial 1 with value: 0.6976141785957737.
[I 2025-05-20 04:27:11,058] Trial 3 finished with value: 0.3243353783231084 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 53, 'learning_rate': 0.2448731739415382, 'n_estimators': 43}. Best is trial 1 with value: 0.6976141785957737.
[I 2025-05-20 04:27:15,345] Tr

Best parameters for LightGBM: {'boosting_type': 'dart', 'num_leaves': 53, 'learning_rate': 0.120577751991469, 'n_estimators': 171}
Best accuracy for LightGBM: 0.703612815269257


In [33]:
print("Optimizing XGBoost...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50)
print("Best parameters for XGBoost:", study_xgb.best_params)
print("Best accuracy for XGBoost:", study_xgb.best_value)

[I 2025-05-20 04:42:30,159] A new study created in memory with name: no-name-f7009983-3f8b-439f-9302-de951325f99b
/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [04:42:30] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Optimizing XGBoost...


[I 2025-05-20 04:42:35,184] Trial 0 finished with value: 0.546603044762554 and parameters: {'n_estimators': 105, 'learning_rate': 0.1903380729028983, 'max_depth': 4}. Best is trial 0 with value: 0.546603044762554.
/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [04:42:35] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2025-05-20 04:42:39,432] Trial 1 finished with value: 0.6840718018632129 and parameters: {'n_estimators': 34, 'learning_rate': 0.09525863055981645, 'max_depth': 12}. Best is trial 1 with value: 0.6840718018632129.
/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [04:42:39] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2025-05-20 04:42:44,701] Trial 2 finished with value: 0.3753237900477164

Best parameters for XGBoost: {'n_estimators': 147, 'learning_rate': 0.11027003733907077, 'max_depth': 13}
Best accuracy for XGBoost: 0.7087480118154965


In [34]:
print("Optimizing CatBoost...")
study_cb = optuna.create_study(direction="maximize")
study_cb.optimize(objective_cb, n_trials=50)
print("Best parameters for CatBoost:", study_cb.best_params)
print("Best accuracy for CatBoost:", study_cb.best_value)

[I 2025-05-20 04:57:00,397] A new study created in memory with name: no-name-5d38e1d2-6d7a-4341-893a-a7bd661ab069


Optimizing CatBoost...


[I 2025-05-20 04:57:11,173] Trial 0 finished with value: 0.5959100204498977 and parameters: {'depth': 6, 'learning_rate': 0.2992980597396313, 'iterations': 112}. Best is trial 0 with value: 0.5959100204498977.
[I 2025-05-20 04:57:27,694] Trial 1 finished with value: 0.5556009997727789 and parameters: {'depth': 10, 'learning_rate': 0.07170598793612809, 'iterations': 66}. Best is trial 0 with value: 0.5959100204498977.
[I 2025-05-20 04:57:40,649] Trial 2 finished with value: 0.4709838673028857 and parameters: {'depth': 5, 'learning_rate': 0.09226312804852464, 'iterations': 163}. Best is trial 0 with value: 0.5959100204498977.
[I 2025-05-20 04:57:53,214] Trial 3 finished with value: 0.6086344012724381 and parameters: {'depth': 10, 'learning_rate': 0.176054380985315, 'iterations': 51}. Best is trial 3 with value: 0.6086344012724381.
[I 2025-05-20 04:57:59,335] Trial 4 finished with value: 0.3461486025903204 and parameters: {'depth': 4, 'learning_rate': 0.05614309861232177, 'iterations': 12

Best parameters for CatBoost: {'depth': 10, 'learning_rate': 0.29936539990030453, 'iterations': 168}
Best accuracy for CatBoost: 0.6927970915700977


In [35]:
print("Optimizing ElasticNet...")
study_en = optuna.create_study(direction="maximize")
study_en.optimize(objective_en, n_trials=50)
print("Best parameters for ElasticNet:", study_en.best_params)
print("Best accuracy for ElasticNet:", study_en.best_value)

[I 2025-05-20 05:18:34,499] A new study created in memory with name: no-name-0f9be77f-02d7-4fbf-b7b2-743af9dd2529
[I 2025-05-20 05:18:34,532] Trial 0 finished with value: 0.05216996137241536 and parameters: {'alpha': 0.08080446019921118, 'l1_ratio': 0.6870339585430013}. Best is trial 0 with value: 0.05216996137241536.
[I 2025-05-20 05:18:34,567] Trial 1 finished with value: 0.029220631674619406 and parameters: {'alpha': 0.6227070973696065, 'l1_ratio': 0.02247860241435462}. Best is trial 0 with value: 0.05216996137241536.
[I 2025-05-20 05:18:34,588] Trial 2 finished with value: 0.0316291751874574 and parameters: {'alpha': 2.4126844964127505, 'l1_ratio': 0.890874717339292}. Best is trial 0 with value: 0.05216996137241536.
[I 2025-05-20 05:18:34,624] Trial 3 finished with value: 0.05944103612815269 and parameters: {'alpha': 0.014044257172378958, 'l1_ratio': 0.19414968005882127}. Best is trial 3 with value: 0.05944103612815269.
[I 2025-05-20 05:18:34,655] Trial 4 finished with value: 0.058

Optimizing ElasticNet...


[I 2025-05-20 05:18:34,717] Trial 6 finished with value: 0.02853896841626903 and parameters: {'alpha': 2.7470387504294553, 'l1_ratio': 0.12082532960475556}. Best is trial 3 with value: 0.05944103612815269.
[I 2025-05-20 05:18:34,752] Trial 7 finished with value: 0.02903885480572597 and parameters: {'alpha': 1.0006563684846954, 'l1_ratio': 0.004443348395521185}. Best is trial 3 with value: 0.05944103612815269.
[I 2025-05-20 05:18:34,839] Trial 8 finished with value: 0.05630538513974097 and parameters: {'alpha': 0.02973024586843065, 'l1_ratio': 0.03735454565465035}. Best is trial 3 with value: 0.05944103612815269.
[I 2025-05-20 05:18:34,866] Trial 9 finished with value: 0.02985685071574642 and parameters: {'alpha': 0.6823143508480444, 'l1_ratio': 0.782474043874693}. Best is trial 3 with value: 0.05944103612815269.
[I 2025-05-20 05:18:34,914] Trial 10 finished with value: 0.059940922517609636 and parameters: {'alpha': 0.011095920110150039, 'l1_ratio': 0.29402630025096016}. Best is trial 1

Best parameters for ElasticNet: {'alpha': 0.016973532191157287, 'l1_ratio': 0.9370422020730639}
Best accuracy for ElasticNet: 0.06053169734151329


In [40]:
# After the data preprocessing and before model training, add PCA analysis
# After the data preprocessing and before model training, add PCA analysis
print("Performing PCA Analysis...")

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Calculate cumulative variance ratio
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

# Find number of components for 90% and 95% variance
n_components_90 = np.argmax(cumulative_variance_ratio >= 0.90) + 1
n_components_95 = np.argmax(cumulative_variance_ratio >= 0.95) + 1

print(f"Number of components needed for 90% variance: {n_components_90}")
print(f"Number of components needed for 95% variance: {n_components_95}")

# Create PCA transformers for 90% and 95% variance
pca_90 = PCA(n_components=n_components_90)
pca_95 = PCA(n_components=n_components_95)

# Transform the data
X_pca_90 = pca_90.fit_transform(X_scaled)
X_pca_95 = pca_95.fit_transform(X_scaled)

# Split the PCA-transformed data
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(
    X_pca_90, y, test_size=0.3, random_state=42
)

X_train_95, X_test_95, y_train_95, y_test_95 = train_test_split(
    X_pca_95, y, test_size=0.3, random_state=42
)

# Plot cumulative variance ratio
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, 'bo-')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% Variance')
plt.axhline(y=0.95, color='g', linestyle='--', label='95% Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('PCA Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.savefig('pca_variance_plot.png')
plt.close()

Performing PCA Analysis...
Number of components needed for 90% variance: 13
Number of components needed for 95% variance: 14


In [41]:
# Function to save model results
def save_model_results(model_name, original_metrics, pca_90_metrics, pca_95_metrics, n_components_90, n_components_95, cumulative_variance_ratio):
    # Create model-specific directory
    model_dir = os.path.join(results_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)
    
    # Save metrics to CSV
    metrics_df = pd.DataFrame({
        'Version': ['Original', 'PCA_90%', 'PCA_95%'],
        'Accuracy': [original_metrics['accuracy'], pca_90_metrics['accuracy'], pca_95_metrics['accuracy']],
        'Precision': [original_metrics['precision'], pca_90_metrics['precision'], pca_95_metrics['precision']],
        'Recall': [original_metrics['recall'], pca_90_metrics['recall'], pca_95_metrics['recall']],
        'F1_Score': [original_metrics['f1'], pca_90_metrics['f1'], pca_95_metrics['f1']]
    })
    metrics_df.to_csv(os.path.join(model_dir, 'metrics.csv'), index=False)
    
    # Save detailed results to text file
    with open(os.path.join(model_dir, 'detailed_results.txt'), 'w') as f:
        f.write(f"Detailed Results for {model_name}\n")
        f.write("=" * (len(model_name) + 20) + "\n\n")
        
        # Original Model Results
        f.write("Original Model Results:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Accuracy: {original_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {original_metrics['precision']:.4f}\n")
        f.write(f"Recall: {original_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {original_metrics['f1']:.4f}\n\n")
        
        # PCA 90% Results
        f.write("PCA 90% Model Results:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Accuracy: {pca_90_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_90_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_90_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_90_metrics['f1']:.4f}\n\n")
        
        # PCA 95% Results
        f.write("PCA 95% Model Results:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Accuracy: {pca_95_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_95_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_95_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_95_metrics['f1']:.4f}\n\n")
        
        # Performance Differences
        f.write("Performance Differences:\n")
        f.write("-" * 20 + "\n")
        f.write("PCA 90% - Original:\n")
        f.write(f"Accuracy: {pca_90_metrics['accuracy'] - original_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_90_metrics['precision'] - original_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_90_metrics['recall'] - original_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_90_metrics['f1'] - original_metrics['f1']:.4f}\n\n")
        
        f.write("PCA 95% - Original:\n")
        f.write(f"Accuracy: {pca_95_metrics['accuracy'] - original_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_95_metrics['precision'] - original_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_95_metrics['recall'] - original_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_95_metrics['f1'] - original_metrics['f1']:.4f}\n")
        
        # Add PCA information
        f.write("\nPCA Information:\n")
        f.write("-" * 20 + "\n")
        f.write(f"90% variance components: {n_components_90}\n")
        f.write(f"95% variance components: {n_components_95}\n")
        f.write(f"90% variance explained: {cumulative_variance_ratio[n_components_90-1]:.4f}\n")
        f.write(f"95% variance explained: {cumulative_variance_ratio[n_components_95-1]:.4f}\n")
    
    # Create and save model-specific visualizations
    # Create and save model-specific visualizations
    plt.figure(figsize=(12, 6))
    metrics = ['accuracy', 'precision', 'recall', 'f1']  # Changed to lowercase to match dictionary keys
    x = np.arange(len(metrics))
    width = 0.25
    
    plt.bar(x - width, [original_metrics[m] for m in metrics], width, label='Original')
    plt.bar(x, [pca_90_metrics[m] for m in metrics], width, label='PCA 90%')
    plt.bar(x + width, [pca_95_metrics[m] for m in metrics], width, label='PCA 95%')
    
    plt.xlabel('Metrics')
    plt.ylabel('Score')
    plt.title(f'{model_name} Performance Comparison')
    plt.xticks(x, [m.capitalize() for m in metrics])  # Capitalize for display
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(model_dir, 'performance_comparison.png'))
    plt.close()
    
    print(f"Saved results for {model_name}")

In [47]:
# Define the evaluate_model function
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evaluate a model and return its accuracy and other metrics.

    Parameters:
    -----------
    model : estimator object
        The model to evaluate
    X_train, X_test : array-like
        Training and test features
    y_train, y_test : array-like
        Training and test labels
    model_name : str
        Name of the model for results storage

    Returns:
    --------
    float
        Accuracy score of the model
    """
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

    return accuracy


# Initialize results dictionary
results = {}

# Define all models to test
models = {
    'RandomForest': RandomForestClassifier(random_state=42, n_estimators=159, max_depth=20),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000, C=1.6548422497253514),
    'KNN': KNeighborsClassifier(n_neighbors=50),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42, max_depth=20),
    'GradientBoosting': GradientBoostingClassifier(random_state=42, n_estimators=199, learning_rate=0.292201125968903),
    'LightGBM': lgb.LGBMClassifier(random_state=42, boosting_type="dart", num_leaves=56, learning_rate=0.15535886290875817, n_estimators=200),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=187, learning_rate=0.06801454667919628, max_depth=14),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0, depth=10, learning_rate=0.27204783453295783, iterations=176),
    #    'ElasticNet': ElasticNet(random_state=42, alpha=0.014298528466932659, l1_ratio=0.9924196737009162)
    'SVM': SVC(random_state=42, probability=True)
}

In [46]:
# Train and evaluate all models
print("\nTraining models...")
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Original model
    accuracy_orig = evaluate_model(model, X_train, X_test, y_train, y_test, name)
    print(f"{name} (Original) Accuracy: {accuracy_orig:.4f}")
    
    # PCA 90%
    accuracy_90 = evaluate_model(model, X_train_90, X_test_90, y_train_90, y_test_90, f"PCA_90_{name}")
    print(f"{name} (PCA 90%) Accuracy: {accuracy_90:.4f}")
    
    # PCA 95%
    accuracy_95 = evaluate_model(model, X_train_95, X_test_95, y_train_95, y_test_95, f"PCA_95_{name}")
    print(f"{name} (PCA 95%) Accuracy: {accuracy_95:.4f}")
    
    # Save results immediately after all versions are evaluated
    save_model_results(
        name,
        results[name],
        results[f"PCA_90_{name}"],
        results[f"PCA_95_{name}"],
        n_components_90,
        n_components_95,
        cumulative_variance_ratio
    )


Training models...

Training RandomForest...
RandomForest (Original) Accuracy: 0.6571
RandomForest (PCA 90%) Accuracy: 0.5037
RandomForest (PCA 95%) Accuracy: 0.5008
Saved results for RandomForest

Training LogisticRegression...


/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


LogisticRegression (Original) Accuracy: 0.2176


/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


LogisticRegression (PCA 90%) Accuracy: 0.2170


...

/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM (PCA 90%) Accuracy: 0.4933


/home/ahmet/Desktop/deneme/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM (PCA 95%) Accuracy: 0.4953
Saved results for LightGBM

Training XGBoost...
XGBoost (Original) Accuracy: 0.7074
XGBoost (PCA 90%) Accuracy: 0.5006
XGBoost (PCA 95%) Accuracy: 0.5023
Saved results for XGBoost

Training CatBoost...
CatBoost (Original) Accuracy: 0.6885
CatBoost (PCA 90%) Accuracy: 0.4594
CatBoost (PCA 95%) Accuracy: 0.4568
Saved results for CatBoost

Training ElasticNet...


ValueError: Classification metrics can't handle a mix of multiclass and continuous targets

In [1]:
# Create comparison DataFrame
comparison_data = []
for model_name in models.keys():
    original_metrics = results[model_name]
    pca_90_metrics = results[f"PCA_90_{model_name}"]
    pca_95_metrics = results[f"PCA_95_{model_name}"]
    
    comparison_data.append({
        'Model': model_name,
        'Original_Accuracy': original_metrics['accuracy'],
        'PCA_90_Accuracy': pca_90_metrics['accuracy'],
        'PCA_95_Accuracy': pca_95_metrics['accuracy'],
        'Accuracy_Difference_90': pca_90_metrics['accuracy'] - original_metrics['accuracy'],
        'Accuracy_Difference_95': pca_95_metrics['accuracy'] - original_metrics['accuracy'],
        'Original_F1': original_metrics['f1'],
        'PCA_90_F1': pca_90_metrics['f1'],
        'PCA_95_F1': pca_95_metrics['f1'],
        'F1_Difference_90': pca_90_metrics['f1'] - original_metrics['f1'],
        'F1_Difference_95': pca_95_metrics['f1'] - original_metrics['f1']
    })

NameError: name 'models' is not defined

In [2]:
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Accuracy_Difference_90', ascending=False)

NameError: name 'pd' is not defined

In [ ]:
comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Accuracy_Difference_90', ascending=False)

In [ ]:
# Print comparison results
print("\nModel Comparison (Original vs PCA):")
print("====================================")
print(comparison_df.to_string(index=False))

In [ ]:
# Plot comparison
plt.figure(figsize=(15, 8))
x = np.arange(len(models))
width = 0.35

plt.bar(x - width/2, comparison_df['Original_Accuracy'], width, label='Original')
plt.bar(x + width/2, comparison_df['PCA_90_Accuracy'], width, label='PCA 90%')
plt.bar(x + width/2 + width, comparison_df['PCA_95_Accuracy'], width, label='PCA 95%')

plt.xlabel('Models')
plt.ylabel('Accuracy')
plt.title('Model Performance Comparison: Original vs PCA')
plt.xticks(x, comparison_df['Model'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.close()

In [ ]:
# After the model comparison section, add code to save results
print("\nSaving results to files...")

# Save comparison DataFrame to CSV
comparison_df.to_csv('model_comparison_results.csv', index=False)
print("Saved comparison results to 'model_comparison_results.csv'")

In [ ]:
# Save detailed results to a text file
with open('detailed_model_results.txt', 'w') as f:
    f.write("Detailed Model Performance Analysis\n")
    f.write("=================================\n\n")
    
    # Write PCA Analysis Results
    f.write("PCA Analysis\n")
    f.write("------------\n")
    f.write(f"Number of components needed for 90% variance: {n_components_90}\n")
    f.write(f"Number of components needed for 95% variance: {n_components_95}\n\n")
    
    # Write Model Comparison Results
    f.write("Model Comparison Results\n")
    f.write("----------------------\n")
    f.write(comparison_df.to_string(index=False))
    f.write("\n\n")
    
    # Write Detailed Results for Each Model
    f.write("Detailed Results for Each Model\n")
    f.write("-----------------------------\n")
    for model_name in models.keys():
        f.write(f"\n{model_name}:\n")
        f.write("-" * (len(model_name) + 1) + "\n")
        
        # Original Model Results
        f.write("Original Model:\n")
        original_metrics = results[model_name]
        f.write(f"Accuracy: {original_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {original_metrics['precision']:.4f}\n")
        f.write(f"Recall: {original_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {original_metrics['f1']:.4f}\n")
        
        # PCA 90% Model Results
        f.write("\nPCA 90% Model:\n")
        pca_90_metrics = results[f"PCA_90_{model_name}"]
        f.write(f"Accuracy: {pca_90_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_90_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_90_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_90_metrics['f1']:.4f}\n")
        
        # PCA 95% Model Results
        f.write("\nPCA 95% Model:\n")
        pca_95_metrics = results[f"PCA_95_{model_name}"]
        f.write(f"Accuracy: {pca_95_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_95_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_95_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_95_metrics['f1']:.4f}\n")
        
        # Performance Differences
        f.write("\nPerformance Differences:\n")
        f.write(f"Accuracy: {pca_90_metrics['accuracy'] - original_metrics['accuracy']:.4f}\n")
        f.write(f"Precision: {pca_90_metrics['precision'] - original_metrics['precision']:.4f}\n")
        f.write(f"Recall: {pca_90_metrics['recall'] - original_metrics['recall']:.4f}\n")
        f.write(f"F1 Score: {pca_90_metrics['f1'] - original_metrics['f1']:.4f}\n")
        f.write("\n" + "="*50 + "\n")

In [ ]:
print("Saved detailed results to 'detailed_model_results.txt'")

In [ ]:
# Save PCA components information
pca_components_df = pd.DataFrame({
    'Component': range(1, len(explained_variance_ratio) + 1),
    'Explained_Variance_Ratio': explained_variance_ratio,
    'Cumulative_Variance_Ratio': cumulative_variance_ratio
})
pca_components_df.to_csv('pca_components_analysis.csv', index=False)
print("Saved PCA components analysis to 'pca_components_analysis.csv'")